# Sort generated clusters
- Compile structures from given directories, grouped by generation method. 
- Isomers (same formula, different morphologies) exhibiting distinct surface areas are highlighted.
- Useful to identifies duplicates accross folders (duplicates *in a given* folder are removed by default).

In [ ]:
import os
import sys
import pandas as pd

# sys.path.append('/path/to/nanocraft/top/directory') # path to directory containing the src folder; uncomment if notebook used outside of Nanocraft directory

from src.nanocraft.cnfg import *
from src.nanocraft.log_cnfg import *
from src.nanocraft.ioxyz import load_xyz
from src.nanocraft.srflyrs import surface_tessellation

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
root_dir = "./nanocrystals" # contains folders such as sphere_Cd, cube_Te, etc.
rows = []

for dirpath, dirnames, filenames in os.walk(root_dir):
    for fname in filenames:
        if fname.lower().endswith(".xyz"):
            filepath = os.path.join(dirpath, fname)
            system_name = os.path.splitext(fname)[0]     # basename without .xyz
            folder_name = os.path.basename(dirpath)      # last element of dirname
            _, _, area, _ = surface_tessellation(load_xyz(filepath, sanity_check=False, center_COM=True))            
            rows.append({"system": system_name, "folder": folder_name, "area": round(area, 1)})

df = pd.DataFrame(rows, columns=["system", "folder", "area"])
grouped_df = (df.groupby("system", as_index=False).agg({"folder": list, "area": list}))
grouped_df["isomers"] = grouped_df["area"].apply(lambda lst: len(set(lst)) > 1)

In [ ]:
grouped_df

In [ ]:
grouped_df[grouped_df["isomers"]]